In [1]:
"""
训练tabFPN Dynamic
"""
import os
import numpy as np
import pandas as pd

from tabpfn import TabPFNRegressor
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr


os.environ["TABPFN_MODEL_CACHE_DIR"] = "/data2/zhoujb/package/tabpfn_ckpt"
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

IN_PATH = "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/"

traits = [
    "Yield_per_plant",
    "1000-grain_weight",
    "Grain_length_to_width_ratio",
    "Heading_date",
    "Plant_height",
    "Grain_per_panicle",
    "Seed_set_rate",
    "Tiller_number"
]

In [ ]:
results = []

for target_col in traits:

    print("Running trait:", target_col)

    train_df = pd.read_table(os.path.join(IN_PATH, f"trainFill-{target_col}.txt"))
    test_df  = pd.read_table(os.path.join(IN_PATH, f"testFill-{target_col}.txt"))

    feature_cols = [c for c in train_df.columns if c.startswith("fea_")]

    # =========
    # 编码 location 和 year
    # =========

    all_loc = pd.concat([train_df["location"], test_df["location"]]).unique()
    all_year = pd.concat([train_df["year"], test_df["year"]]).unique()

    loc2id = {v:i for i,v in enumerate(all_loc)}
    year2id = {v:i for i,v in enumerate(all_year)}

    train_df["location_id"] = train_df["location"].map(loc2id)
    test_df["location_id"]  = test_df["location"].map(loc2id)

    train_df["year_id"] = train_df["year"].map(year2id)
    test_df["year_id"]  = test_df["year"].map(year2id)

    feature_cols = feature_cols + ["location_id","year_id"]

    # =========
    # year-location pairs
    # =========

    pairs = (
        train_df[["year","location"]]
        .drop_duplicates()
        .sort_values(["year","location"])
    )

    pairs = list(pairs.itertuples(index=False, name=None))

    # =========
    # TrainingWhole
    # =========

    X_train_whole = train_df[feature_cols].values
    y_train_whole = train_df[target_col].values


    #model_whole = TabPFNRegressor(
        #device="cpu",
        #random_state=0,
        #n_jobs=-1,
        #ignore_pretraining_limits=True
    #)

    #model_whole.fit(X_train_whole, y_train_whole)


    # =========
    # 循环环境
    # =========

    for i,(year,location) in enumerate(pairs):

        test_data = test_df[
            (test_df["year"]==year) &
            (test_df["location"]==location)
        ]

        if len(test_data) == 0:
            continue

        X_test = test_data[feature_cols].values
        y_test = test_data[target_col].values


        """
        TrainingWhole
        """

        #preds = model_whole.predict(X_test)

        #whole_r2 = r2_score(y_test, preds)
        #whole_rmse = np.sqrt(mean_squared_error(y_test, preds))
        #whole_pcc = pearsonr(y_test, preds)[0]


        """
        TrainingOnebyOne
        """

        train_data = train_df[
            (train_df["year"]==year) &
            (train_df["location"]==location)
        ]

        if len(train_data) < 5:
            continue

        X_train_local = train_data[feature_cols].values
        y_train_local = train_data[target_col].values

        model_one = TabPFNRegressor(
            device="cpu",
            random_state=0,
            n_jobs=-1,
            ignore_pretraining_limits=False
        )

        model_one.fit(X_train_local, y_train_local)

        preds = model_one.predict(X_test)

        one_r2 = r2_score(y_test, preds)
        one_rmse = np.sqrt(mean_squared_error(y_test, preds))
        one_pcc = pearsonr(y_test, preds)[0]


        results.append({

            "Trait":target_col,
            "Round":i+1,
            "year":year,
            "location":location,

            "train_local":len(train_data),
            "test_data":len(test_data),

            #"TrainingWhole_R2":whole_r2,
            #"TrainingWhole_RMSE":whole_rmse,
            #"TrainingWhole_PCC":whole_pcc,

            "TrainingOnebyOne_R2":one_r2,
            "TrainingOnebyOne_RMSE":one_rmse,
            "TrainingOnebyOne_PCC":one_pcc
        })


result_df = pd.DataFrame(results)

print(result_df)

result_df.to_csv("/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/tabpfn_compare_results.csv", index=False)

Running trait: Yield_per_plant


/data2/zhoujb/anaconda3/envs/scRNAML/lib/python3.11/site-packages/tabpfn/regressor.py:459: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  check_cpu_warning(self.device, X)
